In [5]:
from pathlib import Path
import sys

FILE = Path.cwd().parent.resolve()
sys.path.insert(0, str(FILE))

In [6]:
from src.prepare_splits import load_splits, add_pca_features
from src.load_model_data import load_model_data
from src.data_utils import get_Xy

from icecream import ic
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, cross_validate, train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import PolynomialFeatures
from sklearn.ensemble import RandomForestRegressor
import numpy as np
import pandas as pd

In [7]:
splits = load_splits()
splits.keys()

dict_keys(['train', 'test'])

In [8]:
for split_name, data in splits.items():
    print(f'Split: {split_name}, n_samples: {len(data)}')
    print(data.columns)

Split: train, n_samples: 92
Index(['patient', 'tar', 'tir', 'tbr', 'hba1c', 'age', 'gender', 'bmi',
       'height', 'weight', 'smoking_history', 'alcohol_drinker',
       'diabetes_type', 'diabetes_duration', 'cgm_mean', 'cgm_std', 'cgm_min',
       'cgm_max', 'cgm_cv', 'cgm_q25', 'cgm_q75', 'cgm_iqr',
       'glucose_variability', 'n_hypo_events', 'n_hyper_events',
       'rapid_changes', 'mean_excursion', 'pc1', 'pc2', 'pc3'],
      dtype='str')
Split: test, n_samples: 24
Index(['patient', 'tar', 'tir', 'tbr', 'hba1c', 'age', 'gender', 'bmi',
       'height', 'weight', 'smoking_history', 'alcohol_drinker',
       'diabetes_type', 'diabetes_duration', 'cgm_mean', 'cgm_std', 'cgm_min',
       'cgm_max', 'cgm_cv', 'cgm_q25', 'cgm_q75', 'cgm_iqr',
       'glucose_variability', 'n_hypo_events', 'n_hyper_events',
       'rapid_changes', 'mean_excursion', 'pc1', 'pc2', 'pc3'],
      dtype='str')


In [9]:
SEEDS = [0, 1, 2, 3, 4]
FEATURES = ['pc1', 'pc2', 'pc3']
TARGET = 'hba1c'

model_data = load_model_data()

rf_grid = {
    'max_depth': [3, 5, 10, None],
    'min_samples_leaf': [1, 3, 5, 10],
    'max_features': ['sqrt', 1.0],
}

In [10]:
results = []
for seed in SEEDS:
    train_raw, test_raw = train_test_split(model_data, test_size=0.2, random_state=seed)
    train_df, test_df = add_pca_features(train_raw.reset_index(drop=True), test_raw.reset_index(drop=True))

    X_train, y_train = train_df[FEATURES], train_df[TARGET]
    X_test, y_test = test_df[FEATURES], test_df[TARGET]

    lr = LinearRegression().fit(X_train, y_train)

    rf_search = GridSearchCV(
        RandomForestRegressor(n_estimators=200, random_state=seed, n_jobs=-1),
        rf_grid, cv=5, scoring='r2'
    ).fit(X_train, y_train)
    rf = rf_search.best_estimator_

    for name, model in [('linear', lr), ('random_forest', rf)]:
        pred = model.predict(X_test)
        results.append({
            'seed': seed,
            'model': name,
            'r2': r2_score(y_test, pred),
            'rmse': np.sqrt(mean_squared_error(y_test, pred)),
            'mae': mean_absolute_error(y_test, pred),
            'best_params': rf_search.best_params_ if name == 'random_forest' else None,
        })

results_df = pd.DataFrame(results)
print(results_df.round(3))
print(results_df.groupby('model')[['r2', 'rmse', 'mae']].agg(['mean', 'std']).round(3))

   seed          model     r2    rmse     mae  \
0     0         linear  0.048  31.023  22.655   
1     0  random_forest -0.037  32.386  23.474   
2     1         linear  0.325  22.399  19.190   
3     1  random_forest  0.218  24.115  20.760   
4     2         linear  0.149  27.121  19.747   
5     2  random_forest  0.055  28.577  20.817   
6     3         linear  0.354  25.168  20.576   
7     3  random_forest  0.310  26.016  21.715   
8     4         linear  0.301  23.806  17.119   
9     4  random_forest  0.123  26.659  19.715   

                                         best_params  
0                                               None  
1  {'max_depth': 5, 'max_features': 1.0, 'min_sam...  
2                                               None  
3  {'max_depth': 3, 'max_features': 'sqrt', 'min_...  
4                                               None  
5  {'max_depth': 3, 'max_features': 'sqrt', 'min_...  
6                                               None  
7  {'max_depth': 5, 